In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor

from pymargins import GComputation  # 0.4.0: Margins -> GComputation
from pymargins.adapters import SklearnBootstrapAdapter

rng = np.random.default_rng(42)
n = 1000
df = pd.DataFrame({
    "age": rng.normal(50, 10, size=n),
    "female": rng.binomial(1, 0.52, n),
    "treated": rng.binomial(1, 0.40, n),
})
df["age_sq"] = df["age"] ** 2
df["y"] = (
    10.0
    + 0.2 * df["age"]
    + 0.001 * df["age_sq"]
    - 0.5 * df["female"]
    + 0.8 * df["treated"]
    + rng.normal(size=n)
)

In [2]:
X = df[["age", "female", "treated", "age_sq"]]
y = df["y"]
model = LinearRegression()
model.fit(X, y)

adapter = SklearnBootstrapAdapter(model, X_train=X, y_train=y)
m = GComputation(model, adapter=adapter, method="bootstrap", B=200, seed=42)
print(m.predict(atexog={"treated": [0, 1]}).summary())

/home/hunter/Workspace/pymargins/pymargins/estimators/_base.py:176: SoundnessWarning: [tail_count_low] Bootstrap tail count ≈ 5.0 (< 10) at level=0.95: percentile tails are poorly estimated. Increase B or use ci="se".
  self._plan, self._report, self._compiled = compile(


              Graph Result (bootstrap, level=0.95)              
           estimate  std err  statistic  P>|z|  [95% Conf. Int.]
----------------------------------------------------------------
treated=0   22.2613   0.0388    22.2613  0.000  22.1899, 22.3444
treated=1   23.0697   0.0574    23.0697  0.000  22.9531, 23.1762

n = 1000
plan ec7edb6@1 | κ = max inf


In [3]:
model_poly = LinearRegression(fit_intercept=False)
X_poly = pd.DataFrame({
    "age": df["age"],
    "female": df["female"],
    "treated": df["treated"],
    "age_sq": df["age_sq"],
})
model_poly.fit(X_poly, df["y"])

adapter_poly = SklearnBootstrapAdapter(
    model_poly,
    formula="y ~ 0 + age + female + treated + I(age**2)",
    data=df,
    target_name="y",
)
m_poly = GComputation(
    model_poly, adapter=adapter_poly,
    method="bootstrap", B=200, seed=42,
)
slope = m_poly.dydx("age")
print(slope.summary())

/home/hunter/Workspace/pymargins/pymargins/estimators/_base.py:176: SoundnessWarning: [tail_count_low] Bootstrap tail count ≈ 5.0 (< 10) at level=0.95: percentile tails are poorly estimated. Increase B or use ci="se".
  self._plan, self._report, self._compiled = compile(


           Graph Result (bootstrap, level=0.95)           
     estimate  std err  statistic  P>|z|  [95% Conf. Int.]
----------------------------------------------------------
age    0.3140   0.0042     0.3140  0.000    0.3060, 0.3224

n = 1000
plan d09adf3@1 | κ = inf


In [4]:
gbr = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=42)
gbr.fit(X, y)

adapter_gbr = SklearnBootstrapAdapter(gbr, X_train=X, y_train=y)
m_gbr = GComputation(gbr, adapter=adapter_gbr, method="bootstrap", B=100, seed=42)
print(m_gbr.predict(atexog={"treated": [0, 1]}).summary())

/home/hunter/Workspace/pymargins/pymargins/estimators/_base.py:176: SoundnessWarning: [tail_count_low] Bootstrap tail count ≈ 2.5 (< 10) at level=0.95: percentile tails are poorly estimated. Increase B or use ci="se".
  self._plan, self._report, self._compiled = compile(


              Graph Result (bootstrap, level=0.95)              
           estimate  std err  statistic  P>|z|  [95% Conf. Int.]
----------------------------------------------------------------
treated=0   22.2911   0.0401    22.2911  0.000  22.2261, 22.3792
treated=1   23.0330   0.0558    23.0330  0.000  22.9096, 23.1099

n = 1000
plan 77f6671@1 | κ = max inf
